# K_09 – Eigenverbrauchsoptimierung

**Grid-Arbitrage** · Batteriespeicher-Arbitrage im Schweizer Strommarkt (Kür)

**Gruppe:** SC26_Gruppe_2 | **Verantwortlich:** Patrik Neunteufel | **Datum:** März 2026

---

*Netz-Eigenverbrauchsoptimierung als Alternative und Ergänzung zur [Grid-Arbitrage](../organisation/O_02_Glossar.ipynb#g-grid-arbitrage).*


| [← K_08 – Alternative Speicher](K_08_Alternative_Speicher.ipynb) | [↑ Übersicht ↑](../organisation/O_01_Project_Overview.ipynb) | [K_10 – Produktsteckbrief →](K_10_Produkt_Steckbrief.ipynb) |
|:---|:---:|---:|

## Inhaltsverzeichnis<a id='toc_K_09'></a>

1. [Initialisierung](#initialisierung_K_09)
2. [Abgrenzung: Vier Modelle im Vergleich](#abgrenzung-vier-modelle-im-vergleich_K_09)
3. [Modell: Niedertarif-Laden, Hochtarif-Eigenverbrauch](#modell-niedertarif-laden-hochtarif-eigenverbrauch_K_09)
4. [Visualisierungen](#visualisierungen_K_09)
5. [Vergleich: Eigenverbrauch vs. Grid-Arbitrage](#vergleich-eigenverbrauch-vs-grid-arbitrage_K_09)
6. [Netz-Eigenverbrauchsoptimierung](#netz-eigenverbrauchsoptimierung_K_09)
7. [Erweiterung: Hybrid-Modell — Eigenverbrauch + Arbitrage](#erweiterung-hybrid-modell-eigenverbrauch-arbitrage_K_09)
8. [Fazit](#fazit_K_09)


## Initialisierung<a id='initialisierung_K_09'></a><a id='0-setup_K_09'></a>

[↑ Inhaltsverzeichnis](#toc_K_09)

Bibliotheken laden, `../sync/config.json` lesen, Verzeichnispfade setzen.

**Imports und Versionen:**

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
from datetime import datetime

# Versionen anzeigen für Reproduzierbarkeit
print(f"Numpy        Version: {np.__version__}")
print(f"Pandas       Version: {pd.__version__}")
print(f"📅 Zuletzt ausgeführt am: {datetime.now().strftime('%d.%m.%Y um %H:%M:%S')}")


**Setup – Konfiguration & Verzeichnisstruktur:** Lädt `../sync/config.json` (SSOT), setzt Pfade.

In [ ]:
with open('../sync/config.json') as f:
    CFG = json.load(f)

SZ_AKTIV   = CFG['szenarien']['gleichzeitigkeit_aktiv']
_sim_cfg   = CFG['pflicht']['simulation']
SOC_MIN_PCT= _sim_cfg['soc_min_pct']       # 0.05 — SSOT
SOC_MAX_PCT= _sim_cfg['soc_max_pct']       # 0.95 — SSOT
EFFICIENCY = _sim_cfg['efficiency_roundtrip'] # 0.92 — SSOT
EUR_CHF    = CFG.get('eur_chf', 0.97)   # Fixkurs aus ../sync/config.json (SSOT)
FORCE_RELOAD = CFG.get('force_reload', {})  # konventionskonform gelesen
DIR_PROC   = os.path.join('../data', 'processed')
CHARTS_DIR = os.path.join('../output', 'charts', SZ_AKTIV)
os.makedirs(CHARTS_DIR, exist_ok=True)
DPI = CFG['visualisierung']['output_dpi']  # SSOT: ../sync/config.json

# ── Farben & Stil aus ../sync/config.json (SSOT) ─────────────────────────────────────
# Bestehende Variablen (Rückwärtskompatibilität)
_viz        = CFG.get('visualisierung', {}).get('farben', {})
BG_DARK     = _viz.get('bg_dark',    '#0d1117')
BG_PANEL    = _viz.get('bg_panel',   '#141414')
C_PRICE     = _viz.get('c_price',    '#FFA726')
C_LOAD      = _viz.get('c_load',     '#66BB6A')
C_CHARGE    = _viz.get('c_charge',   '#1565C0')
C_FEED      = _viz.get('c_feed',     '#B71C1C')
SEG_COLORS  = _viz.get('seg_colors', ['#42A5F5', '#66BB6A', '#FFA726', '#EF5350'])
C_PRIV, C_GEW, C_IND, C_UTIL = SEG_COLORS

# UI-Strukturfarben
C_ACHSE      = _viz.get('c_achse',      '#aaaaaa')  # Achsenbeschriftungen
C_TICK       = _viz.get('c_tick',       '#bbbbbb')  # Tick-Labels
C_SPINE      = _viz.get('c_spine',      '#333333')  # Achsenrahmen
C_LEGENDE_BG = _viz.get('c_legende_bg', '#111111')  # Legenden-Hintergrund
C_GITTER     = _viz.get('c_gitter',     '#cccccc')  # Gitterlinien

# Funktionale Extrafarben (nur laden was das NB braucht)
C_DISPATCH   = _viz.get('c_dispatch',   '#AB47BC')  # Dispatch-optimal
C_STACKING   = _viz.get('c_stacking',   '#5DCAA5')  # Revenue Stacking
C_SOLAR      = _viz.get('c_solar',      '#FDD835')  # Solar-Ertrag
C_GRENZWERT  = _viz.get('c_amber_dark', '#FF6F00')  # Grenzwert / Warnung
C_CYAN       = _viz.get('c_cyan',       '#26C6DA')  # Flusswasser / Alt. Speicher
C_GRUEN_DARK = _viz.get('c_gruen_dark', '#388E3C')  # Erneuerbare

# Stilkonstanten
_stil               = CFG.get('visualisierung', {}).get('stil', {})
LW                  = _stil.get('linienbreite_standard', 1.5)   # Standard-Linienbreite
LW_DUENN            = _stil.get('linienbreite_duenn',    0.8)   # dünne Linien
LW_DICK             = _stil.get('linienbreite_dick',     2.5)   # dicke Linien
ALPHA_FLAECHE       = _stil.get('alpha_flaeche',         0.12)  # dezente Füllung
ALPHA_FLAECHE_STARK = _stil.get('alpha_flaeche_stark',   0.35)  # Balken / Füllung
ALPHA_LEGENDE       = _stil.get('alpha_legende',         0.30)  # Legenden-BG
ALPHA_GEDAEMPFT     = _stil.get('alpha_linie_gedaempft', 0.55)  # Nebenlinien
FS_TITEL            = _stil.get('schriftgroesse_titel',   13)   # Chart-Titel
FS_ACHSE            = _stil.get('schriftgroesse_achse',   10)   # Achsenbeschr.
FS_TICK             = _stil.get('schriftgroesse_tick',     9)   # Ticks
FS_LEGENDE          = _stil.get('schriftgroesse_legende',  8)   # Legende
FS_KLEIN            = _stil.get('schriftgroesse_klein',    7)   # Annotationen

# matplotlib rcParams — nur stabile, versionsunabhängige Keys (matplotlib >= 3.5)
# axes.titlecolor (3.8+) und axes.grid (stört Karten) bewusst NICHT gesetzt
import matplotlib as mpl
mpl.rcParams.update({
    'figure.facecolor':  BG_DARK,
    'axes.facecolor':    BG_PANEL,
    'axes.edgecolor':    C_SPINE,
    'axes.labelcolor':   C_ACHSE,
    'axes.labelsize':    FS_ACHSE,
    'axes.titlesize':    FS_TITEL,
    'xtick.color':       C_TICK,
    'ytick.color':       C_TICK,
    'xtick.labelsize':   FS_TICK,
    'ytick.labelsize':   FS_TICK,
    'text.color':        'white',
    'lines.linewidth':   LW,
    'legend.facecolor':  C_LEGENDE_BG,
    'legend.framealpha': ALPHA_LEGENDE,
    'legend.fontsize':   FS_LEGENDE,
    'legend.edgecolor':  C_SPINE,
})
print('Farben & Stil geladen.')

C_OWN    = C_LOAD

# Preisdaten laden (aus NB01/NB02)
PRICES_FILE = os.path.join(DIR_PROC, 'ch_spot_prices_clean.csv')
if os.path.exists(PRICES_FILE):
    df_prices = pd.read_csv(PRICES_FILE, parse_dates=['timestamp'])
    df_prices['timestamp'] = pd.to_datetime(df_prices['timestamp'], utc=True)
    df_prices['hour'] = df_prices['timestamp'].dt.hour
    print(f"Preisdaten geladen: {len(df_prices):,} Stunden")
else:
    # Synthetisches Tagesprofil für Demonstration
    print("⚠️  Preisdaten nicht gefunden — NB01 zuerst ausführen. Verwende synthetisches Tagesprofil.")
    hours = np.arange(24)
    base = 80
    profile = (base
               + 15 * np.exp(-((hours - 8)**2) / 8)
               + 25 * np.exp(-((hours - 19)**2) / 6)
               - 20 * np.exp(-((hours - 13)**2) / 10))
    df_prices = pd.DataFrame({'hour': hours, 'price_eur_mwh': profile})

print(f"Setup OK | Charts → {CHARTS_DIR}")

---
## 2. Abgrenzung: Vier Modelle im Vergleich <a id='abgrenzung-vier-modelle-im-vergleich_K_09'></a>

[↑ Inhaltsverzeichnis](#toc_K_09)

| Merkmal | [Grid-Arbitrage](../organisation/O_02_Glossar.ipynb#g-grid-arbitrage) (NB04) | **Netz-Eigenverbrauch (K_09)** | Solar-Eigenverbrauch | **Hybrid: EV + Arbitrage** |
|---|---|---|---|---|
| **Energiequelle Laden** | Netz (günstige Stunden) | **Netz (NT-Fenster)** | Eigene PV-Anlage | Netz (NT) + ggf. PV |
| **Ziel** | Erlös durch Einspeisung | **Kostensenkung Selbstbezug** | Kein Überschuss-Verlust | Kostensenkung + Zusatzerlös |
| **Einspeisung ins Netz** | Ja — zu Spitzenpreisen | **Nein** | Nein (oder minimal) | Ja — Restkapazität |
| **Solar-Anlage nötig** | Nein | **Nein** | Ja | Optional |
| **Erlösmechanismus** | Spread [p75](../organisation/O_02_Glossar.ipynb#g-p25p75) − [p25](../organisation/O_02_Glossar.ipynb#g-p25p75) | **Tarif-Differenz HT − NT** | Eigenverbrauch statt Einspeisung | HT/NT + Spread-Anteil |
| **Regulierung** | Vollständige Marktteilnahme | **Keine Handelserlaubnis nötig** | Keine Handelserlaubnis | Marktteilnahme für Arbitrage-Teil |
| **Zielgruppe** | Industrie / Utility | **Privat / KMU ohne Solar** | Privat / KMU mit Solar | Privat / KMU |
| **Metering** | Smart Meter bidirektional | **Nur Verbrauchsmessung** | Erzeugungsmessung + Verbrauch | Smart Meter bidirektional |
| **Abhängigkeit Nutzerverhalten** | Gering | **Hoch** | Mittel | **Mittel → gering (mit ML)** |
| **Komplexität Steuerung** | Mittel | **Gering** | Gering | Hoch — Optimierungsproblem |
| **Hauptrisiko** | Spread-[Volatilität](../organisation/O_02_Glossar.ipynb#g-volatilitaet) | **Tarif-Änderungen** | Einspeisevergütung | Spread + Verbrauchsvorhersage |

> **Hybrid-Modell (Spalte 4):** Eigenverbrauch hat Vorrang — die Batterie deckt zuerst den
> prognostizierten Eigenbedarf (HT-Stunden). Die verbleibende Kapazität wird für Arbitrage
> genutzt. Dieses Modell **dämpft die Abhängigkeit vom Nutzerverhalten**, da die
> Arbitrage-Komponente unabhängig vom Haushalt dispatcht.  
> Mit **ML-basierter Verbrauchsprognose** lässt sich die optimale Aufteilung täglich
> neu berechnen — der [Dispatch](../organisation/O_02_Glossar.ipynb#g-dispatch)-Algorithmus lernt das individuelle Lastprofil.

> **Warum „Einspeisung = Nein" für den reinen Netz-Eigenverbrauch (K_09-Scope):**  
> Dieses Notebook analysiert bewusst den isolierten Fall ohne Solar und ohne Einspeisung.
> Das Hybrid-Modell ist eine naheliegende Erweiterung, würde aber bidirektionales
> Metering, Marktzulassung und einen ML-Dispatch voraussetzen — deutlich höhere
> Einstiegshürde für Privathaushalte.


---
## 3. Modell: Niedertarif-Laden, Hochtarif-Eigenverbrauch <a id='modell-niedertarif-laden-hochtarif-eigenverbrauch_K_09'></a>

[↑ Inhaltsverzeichnis](#toc_K_09)

### Annahmen

| Parameter | Wert |
|---|---|
| Batteriekapazität | 10 kWh |
| Ladeleistung | 3.7 kW (typisch Heimspeicher) |
| [Round-Trip-Wirkungsgrad](../organisation/O_02_Glossar.ipynb#g-rte) | 92 % |
| Hochtarif (HT) | 08:00–20:00 Uhr |
| Niedertarif (NT) | 20:00–08:00 Uhr + Wochenende |
| HT-Preis | ~0.291 EUR/kWh (0.30 CHF × 0.97 EUR/CHF) |
| NT-Preis | ~0.213 EUR/kWh (0.22 CHF × 0.97 EUR/CHF) |
| Tarif-Differenz | ~0.078 EUR/kWh (0.08 CHF × 0.97) |
| Tagesverbrauch Haushalt | ~8–12 kWh/Tag |

### Dispatch-Logik

```
NT (20:00–08:00): Laden wenn SoC < 90 % und Verbrauch > 0
HT (08:00–20:00): Entladen für Eigenverbrauch, kein Einspeisen
Grenze:           SoC min 5 %, max 95 %
```


In [ ]:
# ── Simulation: Eigenverbrauchsoptimierung ───────────────────────────────────
CAP_KWH   = 10.0
MIN_SOC   = SOC_MIN_PCT  # aus ../sync/config.json (SSOT)
MAX_SOC   = SOC_MAX_PCT  # aus ../sync/config.json (SSOT)
EFFICIENCY= _sim_cfg['efficiency_roundtrip']  # SSOT
LADEL_KW  = 3.7
VERBRAUCH_KWH_TAG = 10

# CH-Haushaltstarife: CHF × EUR_CHF aus ../sync/config.json (SSOT)
HT_PREIS = 0.30 * EUR_CHF  # EUR/kWh  (0.30 CHF)
NT_PREIS = 0.22 * EUR_CHF  # EUR/kWh  (0.22 CHF)

np.random.seed(42)
hours_year = np.arange(8760)
hour_of_day = hours_year % 24
day_of_week = (hours_year // 24) % 7

is_nt = (hour_of_day >= 20) | (hour_of_day < 8) | (day_of_week >= 5)

verbrauch = (VERBRAUCH_KWH_TAG / 24
             * (1 + 0.8 * np.exp(-((hour_of_day - 7.5)**2) / 4)
                  + 1.2 * np.exp(-((hour_of_day - 19)**2) / 5)
                  + np.random.normal(0, 0.05, 8760)))
verbrauch = np.clip(verbrauch, 0, None)

soc    = 0.5
soc_ts = np.zeros(8760)
lad_ts = np.zeros(8760)
eig_ts = np.zeros(8760)
netz_ohne = np.zeros(8760)

for h in range(8760):
    netz_ohne[h] = verbrauch[h]
    if is_nt[h] and soc < MAX_SOC:
        moegl = min(LADEL_KW, (MAX_SOC - soc) * CAP_KWH / EFFICIENCY)
        lade  = min(moegl, 1.0)
        soc  += lade * EFFICIENCY / CAP_KWH
        lad_ts[h] = lade
    elif not is_nt[h] and soc > MIN_SOC:
        moegl_entl = min(verbrauch[h], (soc - MIN_SOC) * CAP_KWH)
        eig_ts[h]  = moegl_entl
        soc       -= moegl_entl / CAP_KWH
    soc_ts[h] = soc

tarif = np.where(is_nt, NT_PREIS, HT_PREIS)
kosten_ohne = netz_ohne * tarif
kosten_mit  = (netz_ohne - eig_ts) * tarif + lad_ts * NT_PREIS

ersparnis_j = kosten_ohne.sum() - kosten_mit.sum()
print(f"Jahresverbrauch:          {netz_ohne.sum():.0f} kWh")
print(f"Davon aus Batterie:       {eig_ts.sum():.0f} kWh ({eig_ts.sum()/netz_ohne.sum()*100:.1f} %)")
print(f"Ladeenergie aus NT-Netz:  {lad_ts.sum():.0f} kWh")
print(f"Kosten ohne Batterie:     {kosten_ohne.sum():.2f} EUR/Jahr")
print(f"Kosten mit Batterie:      {kosten_mit.sum():.2f} EUR/Jahr")
print(f"Jahresersparnis:          {ersparnis_j:.2f} EUR/Jahr")
print(f"CAPEX Batterie (Annahme): ~2 000 EUR (10 kWh × 200 EUR/kWh)")
print(f"Einfache Amortisation:    ~{2000/ersparnis_j:.1f} Jahre")



---
## 4. Visualisierungen <a id='visualisierungen_K_09'></a>

[↑ Inhaltsverzeichnis](#toc_K_09)

**Chart A:** Tagesprofil — Netzlast, Laden (NT), Eigenverbrauch (HT-Substitution) als Flächendiagramm.
**Chart B:** Amortisationskurve Eigenverbrauch vs. [Grid-Arbitrage](../organisation/O_02_Glossar.ipynb#g-grid-arbitrage) im direkten Vergleich.


In [ ]:
# ── Chart A: Tagesprofil — Verbrauch, Laden, Eigenverbrauch ──────────────────
# Durchschnitt über alle Wochentage
avg = lambda arr: np.array([arr[h::24].mean() for h in range(24)])
h24 = np.arange(24)

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
fig.patch.set_facecolor(BG_DARK)
for ax in axes:
    ax.set_facecolor(BG_PANEL); ax.tick_params(colors=C_TICK)
    for sp in ax.spines.values(): sp.set_edgecolor(C_SPINE)

# Panel 1: Tarif + Verbrauch
ax = axes[0]
ax2 = ax.twinx()
ax.bar(h24, avg(verbrauch), color=C_ACHSE, alpha=0.4, label='Verbrauch [kWh/h]')
ax.bar(h24, avg(eig_ts),   color=C_OWN, alpha=0.85, label='Eigenverbrauch Batterie [kWh/h]')
ax.bar(h24, avg(lad_ts),   color=C_CHARGE, alpha=0.75, label='Laden aus NT-Netz [kWh/h]', bottom=0)
ax2.step(h24, np.where(h24 >= 20, NT_PREIS, np.where(h24 < 8, NT_PREIS, HT_PREIS)),
         color=C_PRICE, lw=2, label='Tarif [EUR/kWh]')
ax.set_ylabel('Energie [kWh/h]', color=C_ACHSE)
ax2.set_ylabel('Tarif [EUR/kWh]', color=C_PRICE)
ax2.tick_params(colors=C_PRICE)
ax.set_title('Durchschnittliches Tagesprofil — Eigenverbrauchsoptimierung', color='white', fontweight='bold')
lines1, labs1 = ax.get_legend_handles_labels()
lines2, labs2 = ax2.get_legend_handles_labels()
ax.legend(lines1+lines2, labs1+labs2, fontsize=FS_LEGENDE, framealpha=ALPHA_LEGENDE, facecolor=C_LEGENDE_BG, labelcolor='white')

# Panel 2: SoC-Profil
ax = axes[1]
ax.fill_between(h24, avg(soc_ts)*100, alpha=0.3, color=C_PRIV)
ax.plot(h24, avg(soc_ts)*100, color=C_PRIV, lw=2)
ax.axhspan(5, 10, alpha=0.1, color=C_UTIL, label='SoC-Schutzzone')
ax.axhspan(90, 95, alpha=0.1, color=C_UTIL)
ax.set_ylabel('State of Charge [%]', color=C_ACHSE)
ax.set_xlabel('Tageszeit [h]', color=C_ACHSE)
ax.set_title('Batterie-SoC im Tagesverlauf (Ø)', color='white', fontweight='bold')
ax.set_ylim(0, 100)
ax.legend(fontsize=FS_LEGENDE, framealpha=ALPHA_LEGENDE, facecolor=C_LEGENDE_BG, labelcolor='white')

plt.tight_layout()
p = os.path.join(CHARTS_DIR, 'kuer_k09_tagesprofil.png')
plt.savefig(p, dpi=DPI, bbox_inches='tight', facecolor=BG_DARK)
plt.show(); plt.close()
print(f"Gespeichert: {p}")


**Chart B — Amortisation:** Kumulierter Cashflow Eigenverbrauch vs. [Grid-Arbitrage](../organisation/O_02_Glossar.ipynb#g-grid-arbitrage) —
zeigt welches Modell früher den Break-Even erreicht.


In [ ]:
# ── Chart B: Amortisation ────────────────────────────────────────────────────
CAPEX_EIGENVERBRAUCH = 2000.0  # EUR
OPEX_RATE = CFG['pflicht']['wirtschaftlichkeit']['opex_rate']  # SSOT: ../sync/config.json
opex_j = CAPEX_EIGENVERBRAUCH * OPEX_RATE
netto_j = ersparnis_j - opex_j
jahre   = np.arange(0, 21)
cashflow_kum = -CAPEX_EIGENVERBRAUCH + netto_j * jahre

fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor(BG_DARK); ax.set_facecolor(BG_PANEL)
ax.tick_params(colors=C_TICK)
for sp in ax.spines.values(): sp.set_edgecolor(C_SPINE)

ax.plot(jahre, cashflow_kum, color=C_OWN, lw=LW_DICK, label='Netto-Cashflow (Ersparnis − OPEX)')
ax.axhline(0, color='white', lw=1, ls='--', alpha=0.5)
be = next((j for j, v in zip(jahre, cashflow_kum) if v >= 0), None)
if be:
    ax.axvline(be, color=C_UTIL, lw=1.2, ls=':', label=f'Break-Even: Jahr {be}')
    ax.scatter([be], [0], color=C_UTIL, s=80, zorder=5)

ax.fill_between(jahre, cashflow_kum, 0, where=(cashflow_kum >= 0), alpha=0.15, color=C_OWN)
ax.fill_between(jahre, cashflow_kum, 0, where=(cashflow_kum < 0), alpha=0.15, color=C_UTIL)
ax.set_xlabel('Jahre', color=C_ACHSE)
ax.set_ylabel('Kumulierter Cashflow [EUR]', color=C_ACHSE)
ax.set_title(f'Amortisation Eigenverbrauchsoptimierung\n'
             f'CAPEX {CAPEX_EIGENVERBRAUCH:.0f} EUR | Jahresersparnis {ersparnis_j:.0f} EUR | OPEX {opex_j:.0f} EUR/J',
             color='white', fontweight='bold')
ax.legend(fontsize=FS_TICK, framealpha=ALPHA_LEGENDE, facecolor=C_LEGENDE_BG, labelcolor='white')
ax.grid(True, alpha=ALPHA_FLAECHE)

p = os.path.join(CHARTS_DIR, 'kuer_k09_amortisation.png')
plt.savefig(p, dpi=DPI, bbox_inches='tight', facecolor=BG_DARK)
plt.show(); plt.close()
print(f"Gespeichert: {p}")
print(f"ROI: {netto_j/CAPEX_EIGENVERBRAUCH*100:.1f} %/Jahr")


---
## 5. Vergleich: Eigenverbrauch vs. Grid-Arbitrage <a id='vergleich-eigenverbrauch-vs-grid-arbitrage_K_09'></a>

[↑ Inhaltsverzeichnis](#toc_K_09)

| Kriterium | [Grid-Arbitrage](../organisation/O_02_Glossar.ipynb#g-grid-arbitrage) | Eigenverbrauchsoptimierung |
|---|---|---|
| **Jahreserlös/-ersparnis** | < 97 EUR (Privat, 2023/2024) | ~145–243 EUR |
| **CAPEX-Basis** | ~4 000 EUR (400 EUR/kWh) | ~2 000 EUR (~200 EUR/kWh) |
| **ROI** | < 2 % | ~5–10 % |
| **Komplexität** | Hoch (Marktpreise, Prognosen) | Niedrig (Tarif-Zeitfenster) |
| **Regulierung** | Vollständige Marktzulassung | Keine Sondererlaubnis |
| **Risiko** | Spread-[Volatilität](../organisation/O_02_Glossar.ipynb#g-volatilitaet) | Tarif-Änderungen |
| **Kombination möglich?** | — | ✅ Ja — tagsüber Eigenverbrauch, nachts Restladung für Arbitrage |

**Befund:** Für den Privat-Haushalt ist Eigenverbrauchsoptimierung **wirtschaftlich
überlegen** gegenüber reiner Grid-Arbitrage im Zeitraum 2023/2024.

Die Tarif-Differenz (HT/NT ~0.08 CHF/kWh) ist als Erlösmechanismus
verlässlicher als der schwankende Day-Ahead-Spread.

> **Wichtig:** Modell setzt variable Tarife voraus. Haushalte mit Pauschaltarif
> profitieren nicht. In der Schweiz nehmen HT/NT-Tarife zu, sind aber noch
> nicht universell (Stand 2024/2025).


---
## 6. Netz-Eigenverbrauchsoptimierung <a id='netz-eigenverbrauchsoptimierung_K_09'></a>

[↑ Inhaltsverzeichnis](#toc_K_09)

Bewertet Eigenverbrauchsmodell nach [ROI](../organisation/O_02_Glossar.ipynb#g-roi), Amortisationsdauer und Komplexität;
vergleicht direkt mit dem [Grid-Arbitrage](../organisation/O_02_Glossar.ipynb#g-grid-arbitrage)-Baseline aus NB02.


---
## 7. Erweiterung: Hybrid-Modell — Eigenverbrauch + Arbitrage <a id='erweiterung-hybrid-modell-eigenverbrauch-arbitrage_K_09'></a>

[↑ Inhaltsverzeichnis](#toc_K_09)

### 7.1 Grundidee

Das Hybrid-Modell kombiniert die Verlässlichkeit des Netz-Eigenverbrauchs
(stabile Tarif-Differenz) mit dem Zusatzpotenzial der [Grid-Arbitrage](../organisation/O_02_Glossar.ipynb#g-grid-arbitrage) (Spread-Erlöse).

**Dispatch-Logik:**

```
1. Prognose: Wie viel Energie brauche ich heute zu HT-Zeiten?
2. Reserviere: Ausreichend Kapazität für Eigenverbrauch zurückhalten
3. Arbitrage: Restkapazität nach p25/p75-Logik dispatchen
4. Täglich anpassen: Prognose wird laufend verbessert
```

### 7.2 Warum das Nutzerverhalten-Problem löst

Bei reinem Eigenverbrauch ist der [ROI](../organisation/O_02_Glossar.ipynb#g-roi) stark vom tatsächlichen Verbrauch abhängig —
eine Woche Ferien, ein Homeoffice-Tag mehr, verändert das Ergebnis merklich.

Das Hybrid-Modell dämpft diesen Effekt: Wenn der Eigenverbrauch geringer als
erwartet ausfällt, übernimmt die Arbitrage-Komponente den Resterlös.

| Szenario | Nur Eigenverbrauch | Hybrid |
|---|---|---|
| Normaler Werktag | ✅ Gut | ✅ Gut |
| Ferien (wenig Verbrauch) | ❌ Schlechter ROI | ✅ Arbitrage kompensiert |
| Homeoffice (viel Verbrauch) | ✅ Sehr gut | ✅ Sehr gut |
| Unerwarteter Spread-Tag | ✅ Missed opportunity | ✅ Arbitrage nutzt es |

### 7.3 ML-Dispatch: Den Nutzer «lernen»

Ein einfaches ML-Modell könnte auf dem lokalen Gerät laufen:

- **Input:** Stundenlastprofil der letzten N Tage, Wochentag, Jahreszeit, Wetter (optional)
- **Output:** Prognostizierter stündlicher Verbrauch für den nächsten Tag
- **Optimierung:** Kapazitätsaufteilung Eigenverbrauch / Arbitrage minimiert Gesamtkosten

> **Verweis:** K_06 [Dispatch](../organisation/O_02_Glossar.ipynb#g-dispatch)-Optimierung analysiert den DA-optimalen Dispatch für reine
> Arbitrage. Das Hybrid-Modell wäre eine direkte Erweiterung davon mit einem
> Verbrauchsprognose-Modul.

### 7.4 Wirtschaftliche Einordnung (qualitativ)

| Modell | ROI (grob) | Komplexität | Empfehlung |
|---|---|---|---|
| Reine Arbitrage (NB04) | < 2 % | Mittel | Industrie/Utility |
| Netz-Eigenverbrauch (K_09) | 5–10 % | **Gering** | **Einstieg Privat/KMU** |
| Hybrid (statisch, 70/30) | 6–12 % | Hoch | KMU mit IT-Ressourcen |
| Hybrid (ML-optimiert) | 8–15 % | Sehr hoch | Aggregatoren / [VPP](../organisation/O_02_Glossar.ipynb#g-vpp) |

> **Fazit:** Für den Einstieg ist reiner Netz-Eigenverbrauch am einfachsten.
> Das Hybrid-Modell ist der logische nächste Schritt sobald Smart Meter und
> bidirektionaler Anschluss vorhanden sind — und eine lohnende Basis für
> ML-basierte Dispatch-Optimierung (→ K_06, KI-prädiktiver Dispatch).


---
## 8. Fazit <a id='fazit_K_09'></a>

[↑ Inhaltsverzeichnis](#toc_K_09)

**Netz-Eigenverbrauch** ist für Privathaushalte mit HT/NT-Tarif heute der einfachste
Einstieg: [ROI](../organisation/O_02_Glossar.ipynb#g-roi) 5–10 %, keine Handelserlaubnis, kein bidirektionaler Zähler nötig.

**Das Hybrid-Modell** (Eigenverbrauch + Arbitrage) ist der wirtschaftlich attraktivere
nächste Schritt — es reduziert die Abhängigkeit vom individuellen Nutzerverhalten
und öffnet den Weg zu ML-optimiertem [Dispatch](../organisation/O_02_Glossar.ipynb#g-dispatch).

Die drei Ausbaustufen sind direkt aufeinander aufbauend:

```
Stufe 1 (heute):   Netz-Eigenverbrauch    → einfach, sofort umsetzbar
Stufe 2 (bald):    Hybrid statisch        → wenn Smart Meter vorhanden
Stufe 3 (mittel):  Hybrid ML-optimiert    → wenn Lastprofil-Daten verfügbar
```

> Konkretes Produktbeispiel mit Zahlen für Stufe 1:  
> → [K_10 Produktsteckbrief (7 kW / 14 kWh / 2 000 EUR)](K_10_Produkt_Steckbrief.ipynb)  
> ML-Dispatch-Grundlagen (Stufe 3):  
> → [K_06 Dispatch-Optimierung](K_06_Dispatch_Optimierung.ipynb)


---

In [ ]:
# -- Transfer: Eigenverbrauch Ergebnisse --------------------------------------
_tf_path = '../sync/transfer.json'
_tf = json.loads(open(_tf_path).read() or '{}') if os.path.exists(_tf_path) and os.path.getsize(_tf_path) > 0 else {}
_tf.setdefault('eigenverbrauch', {})
try:
    _tf['eigenverbrauch'].update({
        'roi_ev_privat_pct': round(float(netto_j / CAPEX_EIGENVERBRAUCH * 100), 2),
        'payback_ev_privat_j': round(float(CAPEX_EIGENVERBRAUCH / netto_j), 1) if netto_j > 0 else 999,
        'jahresersparnis_eur': round(float(ersparnis_j), 1),
    })
    print(f"../sync/transfer.json: eigenverbrauch | ROI: {_tf['eigenverbrauch']['roi_ev_privat_pct']}%")
except Exception as _e:
    print(f"../sync/transfer.json: eigenverbrauch nicht geschrieben ({_e})")
with open(_tf_path, 'w') as _f: json.dump(_tf, _f, indent=2, ensure_ascii=False)



In [ ]:
# os bereits in Setup geladen
# ── Abschlusskontrolle ───────────────────────────────────────────────────────
charts_out = [f for f in os.listdir(CHARTS_DIR) if f.startswith('kuer_k09_')]
print("=== Abschlusskontrolle K_09 (Kür) ===")
for f in sorted(charts_out):
    print(f"  ✅  {f}")
print(f"  {len(charts_out)} Chart(s) erzeugt")
print("=== Ende ===")



| [← K_08 – Alternative Speicher](K_08_Alternative_Speicher.ipynb) | [↑ Übersicht](../organisation/O_01_Project_Overview.ipynb) | [K_10 – Produktsteckbrief →](K_10_Produkt_Steckbrief.ipynb) |
|:---|:---:|---:|